In [0]:
from pyspark.sql.functions import *
spark

In [0]:
df1 = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/pyspark_real_time/source/source_data/temp/customers.csv")
df1.show(5)


In [0]:
df2 = spark.read.format("json").option("header", "true").option("inferSchema", "true").load("/Volumes/pyspark_real_time/source/source_data/temp/orders.json")
df2.show(5)

In [0]:
df2.printSchema()

In [0]:
df = df1.join(df2, on="customer_id", how="left")

df.select(
    "customer_id",
    "name",
    "email",
    "age",
    "country",
    "amount",
    "order_id",
    "status"
).show(5, truncate=False)

In [0]:
df_enriched = df.withColumn("order_type",
    when(col("amount") >= 200, "high value"). when(col("amount") >= 100, "medium value").otherwise("low value"))

df_enriched.show(5, truncate=False)

In [0]:
df_enriched.write.format("delta").mode("overwrite").save("/Volumes/pyspark_real_time/source/source_data/temp/enriched_orders")
df_enriched = spark.read.format("delta").load("/Volumes/pyspark_real_time/source/source_data/temp/enriched_orders")
df_enriched.show(5, truncate=False)

## Spark SQL

In [0]:
df1.createOrReplaceTempView("customers")
df2.createOrReplaceTempView("orders")

In [0]:
df = spark.sql(f"""with cust as (select * from customers),
          ord as (select * from orders)
           select cust.customer_id, cust.name, cust.email, cust.age, cust.country, ord.amount, ord.order_id, ord.status from cust left join ord on cust.customer_id = ord.customer_id""")


df.show(5, truncate=False)     

In [0]:
df.createOrReplaceTempView("df_enriched")
df_enriched = spark.sql(f""" select * ,
                        case 
                            when amount >= 200 then "high value"
                            when amount >= 100 then "medium value"
                            else "low value"
                        end as order_type
                        from df_enriched""")

df_enriched.show(5, truncate=False)    